In [ ]:
# GOLD FEATURE BUILDER. Multi-year endogenous baseline.
# Reads multiple silver speed/TTI parquet files, builds one reusable combined gold table,
# and also writes year-specific gold tables.
# No train/val/test split here. No modeling here.

import os
import numpy as np
import pandas as pd

# -----------------------
# Config
# -----------------------
SILVER_FILE_MAP = {
    2022: r"..\..\..\Data\b_processed_data\ritis_speed_tt_5min\I-66_Research_Speed_TT_2022_silver.parquet",
    2023: r"..\..\..\Data\b_processed_data\ritis_speed_tt_5min\I-66_Research_Speed_TT_2023_silver.parquet",
    2024: r"..\..\..\Data\b_processed_data\ritis_speed_tt_5min\I-66_Research_Speed_TT_2024_silver.parquet",
    2025: r"..\..\..\Data\b_processed_data\ritis_speed_tt_5min\I-66_Research_Speed_TT_2025_silver.parquet",
}

GOLD_DIR = r"..\..\..\Data\c_final_tables\baseline_model_gold_table"
COMBINED_GOLD_NAME = "I66_phase1_TTI_features_2022_2025_combined"

timestamp_column_utc = "ts_utc"
timestamp_column_local = "ts_local"
tmc_column = "tmc"
direction_column = "direction"
tti_column = "tti"
confidence_column = "confidence"
road_column = "road"
miles_column = "miles"

data_frequency = "5min"

number_of_lags = 12
rolling_windows = [6, 12]
low_confidence_threshold = 30

build_complete_time_grid = True
write_csv_also = False
write_year_specific_outputs = True

os.makedirs(GOLD_DIR, exist_ok=True)


# -----------------------
# Helper functions
# -----------------------
def ensure_timestamp_is_utc(dataframe, timestamp_column_name):
    dataframe = dataframe.copy()
    series = dataframe[timestamp_column_name]

    if hasattr(series.dtype, "tz") and series.dtype.tz is not None:
        return dataframe

    dataframe[timestamp_column_name] = pd.to_datetime(series, errors="coerce", utc=True)
    return dataframe


def ensure_timestamp_is_datetime(dataframe, timestamp_column_name):
    dataframe = dataframe.copy()

    if timestamp_column_name not in dataframe.columns:
        return dataframe

    dataframe[timestamp_column_name] = pd.to_datetime(dataframe[timestamp_column_name], errors="coerce")
    return dataframe


def coerce_numeric_column(dataframe, column_name):
    dataframe = dataframe.copy()

    if column_name not in dataframe.columns:
        return dataframe

    dataframe[column_name] = pd.to_numeric(dataframe[column_name], errors="coerce")
    return dataframe


def build_complete_five_minute_grid(group_dataframe, timestamp_column_name, frequency):
    group_dataframe = group_dataframe.sort_values(timestamp_column_name).copy()

    if group_dataframe.empty:
        return group_dataframe

    min_time = group_dataframe[timestamp_column_name].min()
    max_time = group_dataframe[timestamp_column_name].max()

    if pd.isna(min_time) or pd.isna(max_time):
        return group_dataframe

    full_index = pd.date_range(min_time, max_time, freq=frequency, tz="UTC")

    group_dataframe = (
        group_dataframe
        .set_index(timestamp_column_name)
        .reindex(full_index)
        .rename_axis(timestamp_column_name)
        .reset_index()
    )

    return group_dataframe


def restore_group_identifiers(group_dataframe, source_group_dataframe, identifier_columns):
    group_dataframe = group_dataframe.copy()

    for col in identifier_columns:
        if col in source_group_dataframe.columns:
            group_dataframe[col] = source_group_dataframe[col].iloc[0]

    for static_column in [road_column, miles_column]:
        if static_column in source_group_dataframe.columns:
            group_dataframe[static_column] = source_group_dataframe[static_column].iloc[0]

    return group_dataframe


def add_calendar_features(group_dataframe):
    group_dataframe = group_dataframe.sort_values(timestamp_column_utc).copy()

    group_dataframe["year"] = group_dataframe[timestamp_column_local].dt.year
    group_dataframe["month_number"] = group_dataframe[timestamp_column_local].dt.month
    group_dataframe["day_of_week_number"] = group_dataframe[timestamp_column_local].dt.dayofweek
    group_dataframe["hour_of_day"] = group_dataframe[timestamp_column_local].dt.hour
    group_dataframe["minute_of_hour"] = group_dataframe[timestamp_column_local].dt.minute
    group_dataframe["is_weekend_flag"] = group_dataframe["day_of_week_number"].isin([5, 6]).astype("Int64")

    group_dataframe["sin_hour"] = np.sin(2 * np.pi * group_dataframe["hour_of_day"] / 24.0)
    group_dataframe["cos_hour"] = np.cos(2 * np.pi * group_dataframe["hour_of_day"] / 24.0)

    group_dataframe["sin_day_of_week"] = np.sin(2 * np.pi * group_dataframe["day_of_week_number"] / 7.0)
    group_dataframe["cos_day_of_week"] = np.cos(2 * np.pi * group_dataframe["day_of_week_number"] / 7.0)

    group_dataframe["sin_month"] = np.sin(2 * np.pi * group_dataframe["month_number"] / 12.0)
    group_dataframe["cos_month"] = np.cos(2 * np.pi * group_dataframe["month_number"] / 12.0)

    return group_dataframe


def add_phase1_endogenous_features(group_dataframe):
    group_dataframe = group_dataframe.sort_values(timestamp_column_utc).copy()

    group_dataframe["tti_missing_flag"] = group_dataframe[tti_column].isna().astype("Int64")

    for lag_index in range(1, number_of_lags + 1):
        group_dataframe[f"tti_lag_{lag_index}"] = group_dataframe[tti_column].shift(lag_index)

    for window_size in rolling_windows:
        group_dataframe[f"tti_rolling_mean_{window_size}"] = (
            group_dataframe[tti_column].shift(1).rolling(window_size, min_periods=window_size).mean()
        )
        group_dataframe[f"tti_rolling_standard_deviation_{window_size}"] = (
            group_dataframe[tti_column].shift(1).rolling(window_size, min_periods=window_size).std()
        )

    group_dataframe["tti_change_1_step"] = group_dataframe["tti_lag_1"] - group_dataframe["tti_lag_2"]
    group_dataframe["tti_absolute_change_1_step"] = group_dataframe["tti_change_1_step"].abs()

    if confidence_column in group_dataframe.columns:
        confidence_values = pd.to_numeric(group_dataframe[confidence_column], errors="coerce")
        group_dataframe["is_low_confidence_flag"] = (confidence_values < low_confidence_threshold).astype("Int64")
    else:
        group_dataframe["is_low_confidence_flag"] = 0

    group_dataframe["target_tti_5min_ahead"] = group_dataframe[tti_column].shift(-1)
    group_dataframe["target_tti_15min_ahead"] = group_dataframe[tti_column].shift(-3)
    group_dataframe["target_tti_30min_ahead"] = group_dataframe[tti_column].shift(-6)

    return group_dataframe


def select_gold_columns(gold_dataframe):
    identifier_columns = [tmc_column]
    if direction_column in gold_dataframe.columns:
        identifier_columns.append(direction_column)

    base_columns = [
        timestamp_column_utc,
        timestamp_column_local,
        road_column,
        miles_column,
        tti_column,
        confidence_column,
        "tti_missing_flag",
        "year",
        "month_number",
        "day_of_week_number",
        "hour_of_day",
        "minute_of_hour",
        "is_weekend_flag",
        "sin_hour",
        "cos_hour",
        "sin_day_of_week",
        "cos_day_of_week",
        "sin_month",
        "cos_month",
        "target_tti_5min_ahead",
        "target_tti_15min_ahead",
        "target_tti_30min_ahead",
    ]

    feature_columns = []
    feature_columns += [f"tti_lag_{k}" for k in range(1, number_of_lags + 1)]
    feature_columns += [f"tti_rolling_mean_{w}" for w in rolling_windows]
    feature_columns += [f"tti_rolling_standard_deviation_{w}" for w in rolling_windows]
    feature_columns += ["tti_change_1_step", "tti_absolute_change_1_step", "is_low_confidence_flag"]

    keep_columns = [c for c in (identifier_columns + base_columns + feature_columns) if c in gold_dataframe.columns]
    gold_dataframe = gold_dataframe[keep_columns].copy()

    return gold_dataframe


def drop_rows_without_model_ready_features(gold_dataframe):
    gold_dataframe = gold_dataframe.copy()

    required_columns = [tti_column]
    required_columns += [f"tti_lag_{k}" for k in range(1, number_of_lags + 1)]
    required_columns += ["target_tti_5min_ahead", "target_tti_15min_ahead", "target_tti_30min_ahead"]

    required_columns = [c for c in required_columns if c in gold_dataframe.columns]

    gold_dataframe = gold_dataframe.dropna(subset=required_columns)
    return gold_dataframe


def build_gold_table_from_single_silver_file(silver_path, source_year):
    silver_dataframe = pd.read_parquet(silver_path)

    required_columns = {tmc_column, timestamp_column_utc, tti_column}
    missing_columns = required_columns - set(silver_dataframe.columns)
    if missing_columns:
        raise ValueError(f"Silver table is missing required columns: {missing_columns}")

    silver_dataframe = ensure_timestamp_is_utc(silver_dataframe, timestamp_column_utc)
    silver_dataframe = ensure_timestamp_is_datetime(silver_dataframe, timestamp_column_local)
    silver_dataframe = coerce_numeric_column(silver_dataframe, tti_column)
    silver_dataframe = coerce_numeric_column(silver_dataframe, confidence_column)
    silver_dataframe = coerce_numeric_column(silver_dataframe, miles_column)

    silver_dataframe = silver_dataframe.dropna(subset=[tmc_column, timestamp_column_utc]).copy()
    silver_dataframe["source_year"] = source_year

    group_columns = [tmc_column]
    if direction_column in silver_dataframe.columns:
        group_columns.append(direction_column)

    gold_parts = []
    for _, group in silver_dataframe.groupby(group_columns, dropna=False):
        current_group = group.sort_values(timestamp_column_utc).copy()

        if build_complete_time_grid:
            grid_group = build_complete_five_minute_grid(current_group, timestamp_column_utc, data_frequency)
            grid_group = restore_group_identifiers(grid_group, current_group, group_columns + ["source_year"])
        else:
            grid_group = current_group.copy()

        if "source_year" not in grid_group.columns:
            grid_group["source_year"] = source_year

        grid_group = add_calendar_features(grid_group)
        featured_group = add_phase1_endogenous_features(grid_group)
        gold_parts.append(featured_group)

    gold_dataframe = pd.concat(gold_parts, ignore_index=True)
    gold_dataframe = select_gold_columns(gold_dataframe)

    if "source_year" not in gold_dataframe.columns:
        gold_dataframe["source_year"] = source_year

    gold_dataframe = drop_rows_without_model_ready_features(gold_dataframe)
    gold_dataframe = gold_dataframe.sort_values([tmc_column, timestamp_column_utc]).reset_index(drop=True)

    return gold_dataframe


def save_gold_outputs(gold_dataframe, output_name):
    parquet_path = os.path.join(GOLD_DIR, f"{output_name}.parquet")
    gold_dataframe.to_parquet(parquet_path, index=False)

    csv_path = None
    if write_csv_also:
        csv_path = os.path.join(GOLD_DIR, f"{output_name}.csv")
        gold_dataframe.to_csv(csv_path, index=False)

    print("Gold table saved:")
    print(" ", parquet_path)
    if csv_path is not None:
        print(" ", csv_path)
    print("Shape:", gold_dataframe.shape)
    print("Columns:", len(gold_dataframe.columns))

    return parquet_path, csv_path


# -----------------------
# Main build
# -----------------------
yearly_gold_tables = []

for source_year, silver_path in SILVER_FILE_MAP.items():
    if not os.path.exists(silver_path):
        print(f"Missing silver file for {source_year}: {silver_path}")
        continue

    print(f"Building gold table for {source_year} ...")
    year_gold_dataframe = build_gold_table_from_single_silver_file(silver_path, source_year)
    yearly_gold_tables.append(year_gold_dataframe)

    if write_year_specific_outputs:
        save_gold_outputs(year_gold_dataframe, f"I66_phase1_TTI_features_{source_year}")

if len(yearly_gold_tables) == 0:
    raise ValueError("No yearly gold tables were built. Check the silver file paths.")

combined_gold_dataframe = pd.concat(yearly_gold_tables, ignore_index=True)
combined_gold_dataframe = combined_gold_dataframe.sort_values(
    [tmc_column, timestamp_column_utc]
).reset_index(drop=True)

save_gold_outputs(combined_gold_dataframe, COMBINED_GOLD_NAME)



Building gold table for 2022 ...
Gold table saved:
  ..\..\Data\c_final_tables\baseline_model_gold_table\I66_phase1_TTI_features_2022.parquet
Shape: (4301417, 44)
Columns: 44
Building gold table for 2023 ...
Gold table saved:
  ..\..\Data\c_final_tables\baseline_model_gold_table\I66_phase1_TTI_features_2023.parquet
Shape: (4306319, 44)
Columns: 44
Building gold table for 2024 ...
Gold table saved:
  ..\..\Data\c_final_tables\baseline_model_gold_table\I66_phase1_TTI_features_2024.parquet
Shape: (4315183, 44)
Columns: 44
Building gold table for 2025 ...
Gold table saved:
  ..\..\Data\c_final_tables\baseline_model_gold_table\I66_phase1_TTI_features_2025.parquet
Shape: (4306185, 44)
Columns: 44
Gold table saved:
  ..\..\Data\c_final_tables\baseline_model_gold_table\I66_phase1_TTI_features_2022_2025_combined.parquet
Shape: (17229104, 44)
Columns: 44


('..\\..\\Data\\c_final_tables\\baseline_model_gold_table\\I66_phase1_TTI_features_2022_2025_combined.parquet',
 None)